# 问题三：影响因素分析

## 目标
分析专业舞者和名人特征（年龄、行业等）对比赛表现的影响。

## 核心任务
1. 识别影响比赛表现的关键因素
2. 量化各因素的影响程度
3. 比较因素对评委打分 vs 观众投票的影响差异

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid')

FIGSIZE_NORMAL = (10, 6)
FIGSIZE_WIDE = (12, 6)
COLORS = {
    'primary': '#4682B4',
    'secondary': '#FF7F50',
    'accent': '#228B22',
    'neutral': '#708090'
}

np.random.seed(42)

In [ ]:
# 读取数据
df = pd.read_csv('../数据预处理/data_processed.csv')
vote_estimates = pd.read_csv('../问题一/vote_estimates.csv')

print(f'选手数据: {len(df)} 行')
print(f'投票估算: {len(vote_estimates)} 行')
print(f'\n可用特征:')
print(df.columns.tolist())

## 第一部分：特征工程

识别可用于分析的特征：
1. **名人特征**: celebrity_age, celebrity_industry, celebrity_homestate
2. **专业舞者**: pro_dancer
3. **赛季特征**: season, voting_method

In [ ]:
# 准备分析数据
# 合并选手基本信息到投票估算数据
contestant_info = df[['contestant', 'season', 'celebrity_age', 'celebrity_industry', 
                       'celebrity_homestate', 'pro_dancer', 'final_rank', 'is_finalist',
                       'overall_avg_score', 'active_weeks']].copy()

# 计算每位选手的平均投票和评分
contestant_votes = vote_estimates.groupby(['contestant', 'season']).agg({
    'estimated_vote_prop': 'mean',
    'total_score': 'mean'
}).reset_index()
contestant_votes.columns = ['contestant', 'season', 'avg_vote_prop', 'avg_score']

# 合并
analysis_df = contestant_info.merge(contestant_votes, on=['contestant', 'season'], how='left')
analysis_df = analysis_df.dropna(subset=['avg_vote_prop', 'avg_score'])

print(f'分析数据: {len(analysis_df)} 位选手')
analysis_df.head()

In [ ]:
# 特征编码
# 行业编码
industry_dummies = pd.get_dummies(analysis_df['celebrity_industry'], prefix='industry')

# 地区编码（简化：是否美国本土）
analysis_df['is_us'] = (analysis_df['celebrity_homestate'] != 'Non-US').astype(int)

# 专业舞者编码
pro_dancer_encoder = LabelEncoder()
analysis_df['pro_dancer_encoded'] = pro_dancer_encoder.fit_transform(analysis_df['pro_dancer'].fillna('Unknown'))

# 合并特征
features_df = pd.concat([analysis_df, industry_dummies], axis=1)

print('行业分布:')
print(analysis_df['celebrity_industry'].value_counts())

In [ ]:
# 定义特征列
numeric_features = ['celebrity_age', 'season', 'is_us']
industry_features = [col for col in features_df.columns if col.startswith('industry_')]

all_features = numeric_features + industry_features
print(f'总特征数: {len(all_features)}')
print(f'特征列表: {all_features}')

## 第二部分：描述性分析

In [ ]:
# 按行业分析表现
industry_performance = analysis_df.groupby('celebrity_industry').agg({
    'avg_score': ['mean', 'std', 'count'],
    'avg_vote_prop': ['mean', 'std'],
    'final_rank': 'mean'
}).round(3)

industry_performance.columns = ['avg_score_mean', 'avg_score_std', 'count', 
                                 'avg_vote_mean', 'avg_vote_std', 'avg_rank']
industry_performance = industry_performance.sort_values('avg_score_mean', ascending=False)

print('='*60)
print('【按行业分析】')
print('='*60)
print(industry_performance.to_string())

In [ ]:
# 年龄与表现的关系
print('\n' + '='*60)
print('【年龄与表现的相关性】')
print('='*60)

age_score_corr = analysis_df['celebrity_age'].corr(analysis_df['avg_score'])
age_vote_corr = analysis_df['celebrity_age'].corr(analysis_df['avg_vote_prop'])
age_rank_corr = analysis_df['celebrity_age'].corr(analysis_df['final_rank'])

print(f'年龄 vs 评委得分: r = {age_score_corr:.3f}')
print(f'年龄 vs 观众投票: r = {age_vote_corr:.3f}')
print(f'年龄 vs 最终排名: r = {age_rank_corr:.3f} (正相关=排名更靠后)')

In [ ]:
# 可视化：行业对评委得分和观众投票的影响
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 图1：行业 vs 评委得分
ax1 = axes[0]
industry_order = industry_performance.index.tolist()
sns.boxplot(data=analysis_df, x='celebrity_industry', y='avg_score', 
            order=industry_order, ax=ax1, palette='Blues_d')
ax1.set_xlabel('Celebrity Industry')
ax1.set_ylabel('Average Judge Score')
ax1.tick_params(axis='x', rotation=45)

# 图2：行业 vs 观众投票
ax2 = axes[1]
sns.boxplot(data=analysis_df, x='celebrity_industry', y='avg_vote_prop', 
            order=industry_order, ax=ax2, palette='Oranges_d')
ax2.set_xlabel('Celebrity Industry')
ax2.set_ylabel('Average Vote Proportion')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('figures/fig1_industry_impact.pdf', bbox_inches='tight')
plt.show()

print('='*60)
print('【图1数据特征】')
print(f'   行业数量: {len(industry_order)}')
print(f'   评委得分最高行业: {industry_order[0]}')
print(f'   评委得分最低行业: {industry_order[-1]}')
print('='*60)

In [ ]:
# 可视化：年龄的影响
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 图1：年龄 vs 评委得分
ax1 = axes[0]
ax1.scatter(analysis_df['celebrity_age'], analysis_df['avg_score'], 
            alpha=0.5, color=COLORS['primary'])
z = np.polyfit(analysis_df['celebrity_age'].dropna(), 
               analysis_df.loc[analysis_df['celebrity_age'].notna(), 'avg_score'], 1)
p = np.poly1d(z)
age_range = np.linspace(analysis_df['celebrity_age'].min(), analysis_df['celebrity_age'].max(), 100)
ax1.plot(age_range, p(age_range), 'r--', linewidth=2, label=f'r={age_score_corr:.3f}')
ax1.set_xlabel('Celebrity Age')
ax1.set_ylabel('Average Judge Score')
ax1.legend()

# 图2：年龄 vs 观众投票
ax2 = axes[1]
ax2.scatter(analysis_df['celebrity_age'], analysis_df['avg_vote_prop'], 
            alpha=0.5, color=COLORS['secondary'])
z2 = np.polyfit(analysis_df['celebrity_age'].dropna(), 
                analysis_df.loc[analysis_df['celebrity_age'].notna(), 'avg_vote_prop'], 1)
p2 = np.poly1d(z2)
ax2.plot(age_range, p2(age_range), 'r--', linewidth=2, label=f'r={age_vote_corr:.3f}')
ax2.set_xlabel('Celebrity Age')
ax2.set_ylabel('Average Vote Proportion')
ax2.legend()

plt.tight_layout()
plt.savefig('figures/fig2_age_impact.pdf', bbox_inches='tight')
plt.show()

print('='*60)
print('【图2数据特征】')
print(f'   年龄范围: {analysis_df["celebrity_age"].min():.0f} - {analysis_df["celebrity_age"].max():.0f}')
print(f'   年龄vs评委得分相关系数: {age_score_corr:.3f}')
print(f'   年龄vs观众投票相关系数: {age_vote_corr:.3f}')
print('='*60)

## 第三部分：回归分析

使用多元回归分析各因素的影响程度

In [ ]:
# 准备回归数据
X = features_df[all_features].copy()
X = X.fillna(X.mean())

y_score = features_df['avg_score']
y_vote = features_df['avg_vote_prop']

# 标准化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'样本数: {len(X)}')
print(f'特征数: {X.shape[1]}')

In [ ]:
# 评委得分回归
model_score = Ridge(alpha=1.0)
model_score.fit(X_scaled, y_score)

score_cv = cross_val_score(model_score, X_scaled, y_score, cv=5, scoring='r2')

print('='*60)
print('【评委得分回归模型】')
print('='*60)
print(f'R² (交叉验证): {score_cv.mean():.3f} (+/- {score_cv.std():.3f})')

# 特征重要性
score_importance = pd.DataFrame({
    'feature': all_features,
    'coefficient': model_score.coef_
}).sort_values('coefficient', key=abs, ascending=False)

print('\n特征重要性 (标准化系数):')
print(score_importance.head(10).to_string(index=False))

In [ ]:
# 观众投票回归
model_vote = Ridge(alpha=1.0)
model_vote.fit(X_scaled, y_vote)

vote_cv = cross_val_score(model_vote, X_scaled, y_vote, cv=5, scoring='r2')

print('='*60)
print('【观众投票回归模型】')
print('='*60)
print(f'R² (交叉验证): {vote_cv.mean():.3f} (+/- {vote_cv.std():.3f})')

# 特征重要性
vote_importance = pd.DataFrame({
    'feature': all_features,
    'coefficient': model_vote.coef_
}).sort_values('coefficient', key=abs, ascending=False)

print('\n特征重要性 (标准化系数):')
print(vote_importance.head(10).to_string(index=False))

In [ ]:
# 使用随机森林获取特征重要性
rf_score = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10)
rf_score.fit(X, y_score)

rf_vote = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10)
rf_vote.fit(X, y_vote)

# 特征重要性对比
importance_comparison = pd.DataFrame({
    'feature': all_features,
    'score_importance': rf_score.feature_importances_,
    'vote_importance': rf_vote.feature_importances_
})

importance_comparison['importance_diff'] = importance_comparison['score_importance'] - importance_comparison['vote_importance']
importance_comparison = importance_comparison.sort_values('score_importance', ascending=False)

print('='*60)
print('【随机森林特征重要性对比】')
print('='*60)
print(importance_comparison.head(10).to_string(index=False))

In [ ]:
# 可视化：特征重要性对比
fig, ax = plt.subplots(figsize=(12, 8))

top_features = importance_comparison.head(10)
x = np.arange(len(top_features))
width = 0.35

bars1 = ax.barh(x - width/2, top_features['score_importance'], width,
                label='Judge Score', color=COLORS['primary'])
bars2 = ax.barh(x + width/2, top_features['vote_importance'], width,
                label='Fan Vote', color=COLORS['secondary'])

ax.set_xlabel('Feature Importance')
ax.set_ylabel('Features')
ax.set_yticks(x)
ax.set_yticklabels(top_features['feature'])
ax.legend()
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('figures/fig3_feature_importance.pdf', bbox_inches='tight')
plt.show()

print('='*60)
print('【图3数据特征】')
print(f'   评委得分R²: {score_cv.mean():.3f}')
print(f'   观众投票R²: {vote_cv.mean():.3f}')
print(f'   最重要特征: {top_features.iloc[0]["feature"]}')
print('='*60)

## 第四部分：专业舞者分析

In [ ]:
# 按专业舞者分析
pro_performance = analysis_df.groupby('pro_dancer').agg({
    'avg_score': ['mean', 'std', 'count'],
    'avg_vote_prop': ['mean', 'std'],
    'final_rank': 'mean',
    'is_finalist': 'mean'
}).round(3)

pro_performance.columns = ['avg_score_mean', 'avg_score_std', 'count', 
                            'avg_vote_mean', 'avg_vote_std', 'avg_rank', 'finalist_rate']
pro_performance = pro_performance.sort_values('count', ascending=False)

print('='*60)
print('【按专业舞者分析】')
print('='*60)
print(pro_performance.head(15).to_string())

In [ ]:
# 计算专业舞者效应
# 使用ANOVA检验专业舞者是否显著影响表现
pro_groups_score = [group['avg_score'].values for name, group in analysis_df.groupby('pro_dancer')]
pro_groups_vote = [group['avg_vote_prop'].values for name, group in analysis_df.groupby('pro_dancer')]

# 过滤掉样本太少的组
pro_groups_score = [g for g in pro_groups_score if len(g) >= 3]
pro_groups_vote = [g for g in pro_groups_vote if len(g) >= 3]

f_score, p_score = stats.f_oneway(*pro_groups_score)
f_vote, p_vote = stats.f_oneway(*pro_groups_vote)

print('\n' + '='*60)
print('【专业舞者效应ANOVA检验】')
print('='*60)
print(f'评委得分: F = {f_score:.2f}, p = {p_score:.4f}')
print(f'观众投票: F = {f_vote:.2f}, p = {p_vote:.4f}')

if p_score < 0.05:
    print('结论: 专业舞者对评委得分有显著影响')
else:
    print('结论: 专业舞者对评委得分无显著影响')

if p_vote < 0.05:
    print('结论: 专业舞者对观众投票有显著影响')
else:
    print('结论: 专业舞者对观众投票无显著影响')

In [ ]:
# 可视化：顶级专业舞者对比
top_pros = pro_performance[pro_performance['count'] >= 5].head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 图1：专业舞者 vs 评委得分
ax1 = axes[0]
ax1.barh(range(len(top_pros)), top_pros['avg_score_mean'], 
         xerr=top_pros['avg_score_std'], color=COLORS['primary'], alpha=0.7)
ax1.set_yticks(range(len(top_pros)))
ax1.set_yticklabels(top_pros.index)
ax1.set_xlabel('Average Judge Score')
ax1.invert_yaxis()

# 图2：专业舞者 vs 决赛进入率
ax2 = axes[1]
ax2.barh(range(len(top_pros)), top_pros['finalist_rate'], 
         color=COLORS['secondary'], alpha=0.7)
ax2.set_yticks(range(len(top_pros)))
ax2.set_yticklabels(top_pros.index)
ax2.set_xlabel('Finalist Rate')
ax2.invert_yaxis()

plt.tight_layout()
plt.savefig('figures/fig4_pro_dancer_impact.pdf', bbox_inches='tight')
plt.show()

print('='*60)
print('【图4数据特征】')
print(f'   显示专业舞者数: {len(top_pros)}')
print(f'   平均评委得分最高: {top_pros.index[0]}')
print(f'   决赛进入率最高: {top_pros.sort_values("finalist_rate", ascending=False).index[0]}')
print('='*60)

## 第五部分：综合分析 - 因素对评委vs观众的影响差异

In [ ]:
# 计算各因素对评委得分和观众投票影响的差异
print('='*70)
print('【因素影响差异分析】')
print('='*70)

# 1. 年龄的差异
print('\n1. 年龄因素:')
print(f'   对评委得分影响: r = {age_score_corr:.3f}')
print(f'   对观众投票影响: r = {age_vote_corr:.3f}')
if abs(age_score_corr) > abs(age_vote_corr):
    print('   → 年龄对评委得分影响更大')
else:
    print('   → 年龄对观众投票影响更大')

# 2. 行业的差异
print('\n2. 行业因素:')
industry_score_var = analysis_df.groupby('celebrity_industry')['avg_score'].mean().var()
industry_vote_var = analysis_df.groupby('celebrity_industry')['avg_vote_prop'].mean().var()
print(f'   行业间评委得分方差: {industry_score_var:.4f}')
print(f'   行业间观众投票方差: {industry_vote_var:.6f}')

# 3. 专业舞者的差异
print('\n3. 专业舞者因素:')
print(f'   对评委得分: F = {f_score:.2f}, p = {p_score:.4f}')
print(f'   对观众投票: F = {f_vote:.2f}, p = {p_vote:.4f}')

In [ ]:
# 保存结果
industry_performance.to_csv('industry_analysis.csv')
pro_performance.to_csv('pro_dancer_analysis.csv')
importance_comparison.to_csv('feature_importance.csv', index=False)

# 汇总结果
results_summary = {
    'total_contestants': len(analysis_df),
    'age_score_corr': age_score_corr,
    'age_vote_corr': age_vote_corr,
    'score_model_r2': score_cv.mean(),
    'vote_model_r2': vote_cv.mean(),
    'pro_dancer_score_f': f_score,
    'pro_dancer_score_p': p_score,
    'pro_dancer_vote_f': f_vote,
    'pro_dancer_vote_p': p_vote,
    'top_industry_score': industry_performance.index[0],
    'bottom_industry_score': industry_performance.index[-1]
}

pd.DataFrame([results_summary]).to_csv('results_summary.csv', index=False)
print('\n结果文件已保存')

In [ ]:
# ============================================================
# 问题三建模结果汇总
# ============================================================

print('\n' + '='*70)
print('【问题三建模结果汇总】')
print('='*70)

print('\n1. 年龄影响')
print(f'   年龄 vs 评委得分: r = {age_score_corr:.3f}')
print(f'   年龄 vs 观众投票: r = {age_vote_corr:.3f}')
print(f'   年龄范围: {analysis_df["celebrity_age"].min():.0f} - {analysis_df["celebrity_age"].max():.0f}')

print('\n2. 行业影响')
print(f'   评委得分最高: {industry_performance.index[0]} ({industry_performance.iloc[0]["avg_score_mean"]:.2f})')
print(f'   评委得分最低: {industry_performance.index[-1]} ({industry_performance.iloc[-1]["avg_score_mean"]:.2f})')

print('\n3. 专业舞者影响')
print(f'   ANOVA (评委得分): F = {f_score:.2f}, p = {p_score:.4f}')
print(f'   ANOVA (观众投票): F = {f_vote:.2f}, p = {p_vote:.4f}')

print('\n4. 模型拟合度')
print(f'   评委得分 R²: {score_cv.mean():.3f}')
print(f'   观众投票 R²: {vote_cv.mean():.3f}')

print('\n5. 生成的图片')
figures = [
    'fig1_industry_impact.pdf',
    'fig2_age_impact.pdf',
    'fig3_feature_importance.pdf',
    'fig4_pro_dancer_impact.pdf'
]
for i, fig_name in enumerate(figures, 1):
    print(f'   图{i}: {fig_name}')

print('\n' + '='*70)